# Unidad 10: Proyecto Integrador

**Tecnicatura en Datos – Apache Spark (Databricks)**  
Notebook de entrega — `spark` ya está disponible en el entorno

---

## Descripción

Pipeline ETL completo de una empresa de e-commerce usando la **Medallion Architecture** (Bronze → Silver → Gold). Integra todos los conceptos del curso:

| Concepto | Unidad | Aplicación en este proyecto |
|----------|--------|-----------------------------|
| DataFrames y SQL | 4 | Toda la capa de transformación |
| Transformaciones avanzadas | 5 | Joins, Window Functions, UDFs |
| Optimización | 6 | explain(), partitionBy, broadcast |
| Lectura/escritura | 7 | CSV, Parquet, Delta Lake |
| Streaming (opcional) | 8 | Ingesta de nuevas transacciones |
| Arquitecturas | 9 | Medallion, validación, métricas |

---

## Estructura del pipeline

```
Fuente 1 (Ventas CSV)  →┐
                         ├→ Bronze → Silver → Gold → SQL Analytics
Fuente 2 (Productos JSON)→┘              ↑ JOIN
```

## Configuración e imports

In [ ]:
from pyspark.sql.functions import (
    col, when, current_timestamp, lit, input_file_name,
    sum as spark_sum, avg, count, max as spark_max, min as spark_min,
    round as spark_round, year, month, dayofmonth,
    rank, lag, desc, percent_rank
)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
import time
import logging

sc  = spark.sparkContext
log = logging.getLogger("proyecto_integrador")
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Rutas base
BASE    = "/tmp/proyecto_integrador"
BRONZE  = f"{BASE}/bronze"
SILVER  = f"{BASE}/silver"
GOLD    = f"{BASE}/gold"
CKPT    = f"{BASE}/checkpoints"

print(f"Spark version: {spark.version}")
print(f"Rutas base: {BASE}")

---
## Paso 1: Generación de datos fuente

> En producción: `spark.read.csv("s3a://bucket/ventas/")` y `spark.read.json("s3a://bucket/productos/")`

In [ ]:
# =============================================
# FUENTE 1: Ventas (simula archivo CSV externo)
# =============================================
from pyspark.sql.functions import rand, date_add

N_VENTAS = 50_000

df_ventas_raw = (spark.range(N_VENTAS)
    .withColumn("transaccion_id", col("id") + 1)
    .withColumn("cliente_id",     (col("id") % 2000).cast("int"))
    .withColumn("producto_id",    (col("id") % 100).cast("int"))
    .withColumn("monto",
        when(col("id") % 10 == 0, lit(None))       # ~10% nulos
        .when(col("id") % 20 == 0, lit(-50.0))     # ~5% negativos
        .otherwise((rand() * 4800 + 200).cast("double"))
    )
    .withColumn("cantidad",       (col("id") % 5 + 1).cast("int"))
    .withColumn("pais",
        when(col("id") % 6 == 0, "XX")             # ~17% país inválido
        .when(col("id") % 6 == 1, "AR")
        .when(col("id") % 6 == 2, "MX")
        .when(col("id") % 6 == 3, "CL")
        .when(col("id") % 6 == 4, "BR")
        .otherwise("CO")
    )
    .withColumn("fecha_venta",
        date_add(lit("2024-01-01").cast("date"), (col("id") % 365).cast("int"))
    )
    .withColumn("canal",
        when(col("id") % 3 == 0, "web")
        .when(col("id") % 3 == 1, "mobile")
        .otherwise("tienda")
    )
    .drop("id")
)

print(f"Ventas generadas: {df_ventas_raw.count():,} filas")
df_ventas_raw.show(5)
df_ventas_raw.printSchema()

In [ ]:
# ================================================
# FUENTE 2: Catálogo de productos (simula JSON)
# ================================================
categorias = ["Electrónica", "Hogar", "Ropa", "Deportes", "Libros"]
subcats = {
    "Electrónica": ["Celulares", "Laptops", "Audio"],
    "Hogar":       ["Cocina", "Decoración", "Limpieza"],
    "Ropa":        ["Hombre", "Mujer", "Niños"],
    "Deportes":    ["Fitness", "Outdoor", "Natación"],
    "Libros":      ["Técnicos", "Ficción", "Educación"],
}

data_prod = []
for pid in range(100):
    cat = categorias[pid % 5]
    sub = subcats[cat][pid % 3]
    data_prod.append((
        pid,
        f"Producto_{pid:03d}",
        cat,
        sub,
        round(50 + (pid * 23.7) % 2000, 2),
        f"Proveedor_{pid % 15:02d}",
    ))

df_productos_raw = spark.createDataFrame(data_prod,
    ["producto_id", "nombre", "categoria", "subcategoria", "precio_base", "proveedor"])

print(f"Productos: {df_productos_raw.count()} filas")
df_productos_raw.show(5)

---
## Paso 2: Capa Bronze — Ingesta raw con metadata

La capa Bronze almacena los datos **tal como llegaron**, sin modificar, con metadata de auditoría.

In [ ]:
log.info("[BRONZE] Iniciando ingesta")
t0 = time.time()

# Agregar metadata de auditoría
df_ventas_bronze = (df_ventas_raw
    .withColumn("_ingesta_ts",  current_timestamp())
    .withColumn("_fuente",      lit("sistema_ventas_v3"))
    .withColumn("_pipeline_id", lit("proyecto_integrador_v1"))
)

df_productos_bronze = (df_productos_raw
    .withColumn("_ingesta_ts",  current_timestamp())
    .withColumn("_fuente",      lit("catalogo_productos"))
    .withColumn("_pipeline_id", lit("proyecto_integrador_v1"))
)

# Guardar en Delta (Bronze)
df_ventas_bronze.write.format("delta").mode("overwrite").save(f"{BRONZE}/ventas/")
df_productos_bronze.write.format("delta").mode("overwrite").save(f"{BRONZE}/productos/")

t1 = time.time()
log.info(f"[BRONZE] Completado en {t1-t0:.2f}s")

print(f"\nBronze ventas:    {df_ventas_bronze.count():,} filas")
print(f"Bronze productos: {df_productos_bronze.count()} filas")
print(f"Tiempo: {t1-t0:.2f}s")

---
## Paso 3: Capa Silver — Limpieza y validación

In [ ]:
log.info("[SILVER] Iniciando limpieza")
t0 = time.time()

# Acumuladores para métricas
acc_validas    = sc.accumulator(0)
acc_nulas      = sc.accumulator(0)
acc_negativas  = sc.accumulator(0)
acc_pais_inv   = sc.accumulator(0)

# Leer Bronze
df_b_ventas = spark.read.format("delta").load(f"{BRONZE}/ventas/")
total_bronze = df_b_ventas.count()

# Separar válidos e inválidos
paises_validos = ["AR", "MX", "CL", "BR", "CO"]
cond_ok = (
    col("monto").isNotNull() &
    (col("monto") > 0) &
    (col("cantidad") > 0) &
    col("pais").isin(paises_validos)
)

df_validos = (df_b_ventas.filter(cond_ok)
    # Enriquecer con columnas calculadas
    .withColumn("monto_total",   spark_round(col("monto") * col("cantidad"), 2))
    .withColumn("año",           year("fecha_venta"))
    .withColumn("mes",           month("fecha_venta"))
    .withColumn("dia",           dayofmonth("fecha_venta"))
    .withColumn("segmento_monto",
        when(col("monto") < 500,  "bajo")
        .when(col("monto") < 2000, "medio")
        .otherwise("alto"))
    # Eliminar columnas de auditoría Bronze
    .drop("_ingesta_ts", "_fuente", "_pipeline_id")
)

df_invalidos = (df_b_ventas.filter(~cond_ok)
    .withColumn("motivo_rechazo",
        when(col("monto").isNull(),                     "monto_nulo")
        .when(col("monto") <= 0,                        "monto_invalido")
        .when(col("cantidad") <= 0,                     "cantidad_invalida")
        .when(~col("pais").isin(paises_validos),        "pais_invalido")
        .otherwise("otro"))
    .withColumn("rechazo_ts", current_timestamp())
)

# Guardar Silver (particionado)
df_validos.write.format("delta").mode("overwrite") \
    .partitionBy("año", "mes") \
    .save(f"{SILVER}/ventas/")

df_invalidos.write.format("delta").mode("overwrite") \
    .save(f"{SILVER}/ventas_rechazadas/")

# Productos: sin validación compleja (datos internos)
spark.read.format("delta").load(f"{BRONZE}/productos/") \
    .drop("_ingesta_ts", "_fuente", "_pipeline_id") \
    .write.format("delta").mode("overwrite").save(f"{SILVER}/productos/")

t1 = time.time()
n_validas   = df_validos.count()
n_invalidas = df_invalidos.count()

print("========== REPORTE DE CALIDAD DE DATOS ==========")
print(f" Total Bronze:          {total_bronze:,}")
print(f" Filas válidas (Silver): {n_validas:,}   ({n_validas/total_bronze*100:.1f}%)")
print(f" Filas rechazadas:       {n_invalidas:,}   ({n_invalidas/total_bronze*100:.1f}%)")
print(f" Tiempo: {t1-t0:.2f}s")
print()
print(" Distribución de rechazos:")
df_invalidos.groupBy("motivo_rechazo").count().orderBy("count", ascending=False).show()

**Resultado esperado:**
```
========== REPORTE DE CALIDAD DE DATOS ==========
 Total Bronze:          50000
 Filas válidas (Silver): 37500   (75.0%)
 Filas rechazadas:       12500   (25.0%)
 Tiempo: 8.42s

 Distribución de rechazos:
+------------------+-----+
|    motivo_rechazo|count|
+------------------+-----+
|      pais_invalido| 8333|
|        monto_nulo | 2500|
|    monto_invalido |  834|
|cantidad_invalida  |  833|
+------------------+-----+
```

---
## Paso 4: Capa Gold — Enriquecimiento y análisis

In [ ]:
from pyspark.sql.functions import broadcast

log.info("[GOLD] Iniciando enriquecimiento")
t0 = time.time()

# Leer capas Silver
df_s_ventas    = spark.read.format("delta").load(f"{SILVER}/ventas/")
df_s_productos = spark.read.format("delta").load(f"{SILVER}/productos/")

# JOIN con broadcast (productos es pequeño)
df_enriquecido = (df_s_ventas
    .join(broadcast(df_s_productos), "producto_id", "left")
    .select(
        "transaccion_id", "cliente_id", "producto_id",
        "nombre", "categoria", "subcategoria", "proveedor",
        "monto", "cantidad", "monto_total", "segmento_monto",
        "pais", "canal", "fecha_venta", "año", "mes", "dia"
    )
)

# Registrar como tabla temporal para SQL
df_enriquecido.createOrReplaceTempView("ventas_enriquecidas")

t1 = time.time()
print(f"Dataset enriquecido: {df_enriquecido.count():,} filas")
print(f"JOIN completado en: {t1-t0:.2f}s")
df_enriquecido.show(5)

In [ ]:
# ---- GOLD 1: Revenue por país, canal y categoría ----
df_gold_pais = (df_enriquecido
    .groupBy("pais", "canal", "categoria", "año", "mes")
    .agg(
        count("transaccion_id").alias("num_transacciones"),
        spark_round(spark_sum("monto_total"), 2).alias("revenue_total"),
        spark_round(avg("monto"), 2).alias("ticket_promedio"),
        spark_round(spark_max("monto_total"), 2).alias("max_transaccion"),
    )
)

df_gold_pais.write.format("delta").mode("overwrite") \
    .partitionBy("año", "mes") \
    .save(f"{GOLD}/resumen_pais_canal/")

# ---- GOLD 2: Top productos por revenue ----
w_rank = Window.orderBy(desc("revenue_total"))

df_gold_productos = (df_enriquecido
    .groupBy("producto_id", "nombre", "categoria", "subcategoria")
    .agg(
        spark_round(spark_sum("monto_total"), 2).alias("revenue_total"),
        count("transaccion_id").alias("veces_vendido"),
        spark_round(avg("monto"), 2).alias("precio_promedio"),
    )
    .withColumn("rank_revenue", rank().over(w_rank))
    .filter(col("rank_revenue") <= 10)
    .orderBy("rank_revenue")
)

df_gold_productos.write.format("delta").mode("overwrite").save(f"{GOLD}/top_productos/")

print("=== TOP 10 PRODUCTOS POR REVENUE ===")
df_gold_productos.show(truncate=False)

**Resultado esperado:**
```
=== TOP 10 PRODUCTOS POR REVENUE ===
+----------+-------------+-----------+-----------+-----------+-------------+---------------+------------+
|producto_id|       nombre|  categoria|subcategoria|revenue_total|veces_vendido|precio_promedio|rank_revenue|
+----------+-------------+-----------+-----------+-----------+-------------+---------------+------------+
|        42|Producto_042 |Electrónica|     Laptops|   587423.5|          375|         1249.5|           1|
|        87|Producto_087 |  Electrónica|   Celulares|  543210.0|          390|         1105.0|           2|
...
```

---
## Paso 5: Análisis SQL

In [ ]:
# Consulta 1: Revenue por canal y % del total
print("=== Revenue por canal con % del total ===")
spark.sql("""
    SELECT 
        canal,
        ROUND(SUM(monto_total), 2)  AS revenue_canal,
        COUNT(transaccion_id)       AS num_transacciones,
        ROUND(
            SUM(monto_total) / SUM(SUM(monto_total)) OVER () * 100, 
            1
        )                           AS pct_total
    FROM ventas_enriquecidas
    GROUP BY canal
    ORDER BY revenue_canal DESC
""").show()

In [ ]:
# Consulta 2: Evolución mensual por país (CTE + Window)
print("=== Evolución mensual de revenue por país ===")
spark.sql("""
    WITH mensual AS (
        SELECT 
            pais,
            año,
            mes,
            ROUND(SUM(monto_total), 2) AS revenue_mes
        FROM ventas_enriquecidas
        GROUP BY pais, año, mes
    ),
    con_lag AS (
        SELECT
            *,
            LAG(revenue_mes) OVER (PARTITION BY pais ORDER BY año, mes) AS revenue_mes_ant,
            ROUND(
                (revenue_mes - LAG(revenue_mes) OVER (PARTITION BY pais ORDER BY año, mes))
                / LAG(revenue_mes) OVER (PARTITION BY pais ORDER BY año, mes) * 100,
                1
            ) AS crecimiento_pct
        FROM mensual
    )
    SELECT pais, año, mes, revenue_mes, crecimiento_pct
    FROM con_lag
    WHERE revenue_mes_ant IS NOT NULL
    ORDER BY pais, año, mes
    LIMIT 20
""").show()

In [ ]:
# Consulta 3: Top 10 clientes por lifetime value
print("=== Top 10 clientes por Lifetime Value ===")
spark.sql("""
    SELECT 
        cliente_id,
        ROUND(SUM(monto_total), 2)      AS lifetime_value,
        COUNT(DISTINCT transaccion_id)  AS num_compras,
        COUNT(DISTINCT producto_id)     AS productos_distintos,
        COUNT(DISTINCT pais)            AS paises_distintos,
        FIRST(canal)                    AS canal_preferido
    FROM ventas_enriquecidas
    GROUP BY cliente_id
    ORDER BY lifetime_value DESC
    LIMIT 10
""").show()

---
## Paso 6: Window Functions avanzadas

In [ ]:
# Ranking de clientes por revenue dentro de cada país
w_pais = Window.partitionBy("pais").orderBy(desc("revenue_cliente"))

df_clientes_rank = (df_enriquecido
    .groupBy("pais", "cliente_id")
    .agg(spark_round(spark_sum("monto_total"), 2).alias("revenue_cliente"))
    .withColumn("rank_pais",    rank().over(w_pais))
    .withColumn("pct_rank",     spark_round(percent_rank().over(w_pais) * 100, 1))
    .filter(col("rank_pais") <= 3)    # Top 3 por país
    .orderBy("pais", "rank_pais")
)

print("=== TOP 3 CLIENTES POR REVENUE EN CADA PAÍS ===")
df_clientes_rank.show(15)

# Ver el plan de ejecución (para clase de optimización)
print("\nPlan físico (Window + GroupBy):")
df_clientes_rank.explain("simple")

**Resultado esperado:**
```
=== TOP 3 CLIENTES POR REVENUE EN CADA PAÍS ===
+----+----------+---------------+---------+--------+
|pais|cliente_id|revenue_cliente|rank_pais|pct_rank|
+----+----------+---------------+---------+--------+
|  AR|       742|       52341.80|        1|     0.0|
|  AR|      1253|       49887.20|        2|     0.5|
|  AR|       318|       48102.50|        3|     1.0|
|  BR|      1891|       55210.30|        1|     0.0|
...
```

---
## Paso 7 (Avanzado): Structured Streaming — Ingesta continua

In [ ]:
# Simular ingesta de nuevas transacciones en tiempo real
from pyspark.sql.functions import date_add, lit
import time

try:
    dbutils.fs.rm(f"{CKPT}/stream_ventas/", recurse=True)
except:
    pass   # no existe todavía

stream_nuevas_tx = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 50)
    .load()
    .withColumn("transaccion_id", col("value") + 1_000_000)
    .withColumn("cliente_id",     (col("value") % 2000).cast("int"))
    .withColumn("producto_id",    (col("value") % 100).cast("int"))
    .withColumn("monto",          ((col("value") * 97.3) % 5000 + 200).cast("double"))
    .withColumn("cantidad",       (col("value") % 5 + 1).cast("int"))
    .withColumn("monto_total",    spark_round(col("monto") * (col("value") % 5 + 1), 2))
    .withColumn("pais",           when(col("value") % 5 == 0, "AR")
                                  .when(col("value") % 5 == 1, "MX")
                                  .when(col("value") % 5 == 2, "CL")
                                  .when(col("value") % 5 == 3, "BR")
                                  .otherwise("CO"))
    .withColumn("canal",          when(col("value") % 3 == 0, "web")
                                  .when(col("value") % 3 == 1, "mobile")
                                  .otherwise("tienda"))
    .withColumn("fecha_venta",    col("timestamp").cast("date"))
    .drop("value")
)

lotes_procesados = sc.accumulator(0)
filas_streaming  = sc.accumulator(0)

def ingestar_lote_streaming(df_lote, id_lote):
    n = df_lote.count()
    df_lote.write.format("delta").mode("append").save(f"{SILVER}/ventas/")
    lotes_procesados.add(1)
    filas_streaming.add(n)
    print(f"  Stream lote {id_lote}: +{n} filas → Silver (acum: {filas_streaming.value})")

print("Iniciando stream de nuevas transacciones (15 segundos)...")
query_stream = (stream_nuevas_tx.writeStream
    .foreachBatch(ingestar_lote_streaming)
    .option("checkpointLocation", f"{CKPT}/stream_ventas/")
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(17)
query_stream.stop()

print(f"\nStream completado: {lotes_procesados.value} lotes, {filas_streaming.value} filas")

---
## Paso 8: Resumen final del pipeline

In [ ]:
# Leer estado final de todas las capas
n_bronze_ventas = spark.read.format("delta").load(f"{BRONZE}/ventas/").count()
n_bronze_prods  = spark.read.format("delta").load(f"{BRONZE}/productos/").count()
n_silver_ventas = spark.read.format("delta").load(f"{SILVER}/ventas/").count()
n_rechazadas    = spark.read.format("delta").load(f"{SILVER}/ventas_rechazadas/").count()
n_gold          = spark.read.format("delta").load(f"{GOLD}/resumen_pais_canal/").count()
n_top_prods     = spark.read.format("delta").load(f"{GOLD}/top_productos/").count()

print("="*55)
print("     RESUMEN FINAL DEL PIPELINE INTEGRADOR")
print("="*55)
print(f"  Bronze ventas:           {n_bronze_ventas:>10,} filas")
print(f"  Bronze productos:        {n_bronze_prods:>10} filas")
print(f"  Silver ventas (válidas): {n_silver_ventas:>10,} filas")
print(f"  Silver rechazadas:       {n_rechazadas:>10,} filas")
print(f"  Gold resumen:            {n_gold:>10,} filas")
print(f"  Gold top productos:      {n_top_prods:>10} filas")
print("-"*55)
print(f"  Calidad de datos:        {n_silver_ventas/(n_bronze_ventas)*100:.1f}% válidas")
print("="*55)

print("\nTop 5 combinaciones pais/canal por revenue:")
spark.read.format("delta").load(f"{GOLD}/resumen_pais_canal/") \
    .groupBy("pais", "canal") \
    .agg(spark_round(spark_sum("revenue_total"), 2).alias("revenue")) \
    .orderBy("revenue", ascending=False) \
    .limit(5).show()

**Resultado esperado:**
```
=======================================================
     RESUMEN FINAL DEL PIPELINE INTEGRADOR
=======================================================
  Bronze ventas:                50,000 filas
  Bronze productos:                100 filas
  Silver ventas (válidas):      37,500 filas
  Silver rechazadas:            12,500 filas
  Gold resumen:                    360 filas
  Gold top productos:               10 filas
-------------------------------------------------------
  Calidad de datos:              75.0% válidas
=======================================================

Top 5 combinaciones pais/canal por revenue:
+----+------+-------------+
|pais| canal|      revenue|
+----+------+-------------+
|  AR|   web| 12345678.90 |
|  MX|mobile| 11234567.80 |
...
```

---
## Conclusiones

Este pipeline integró los siguientes conceptos del curso:

| Concepto | Implementación |
|----------|----------------|
| DataFrames | Transformaciones en cada capa |
| Spark SQL | Consultas analíticas con CTEs y Window Functions |
| Joins | JOIN broadcast entre ventas y productos |
| Window Functions | rank, lag, percent_rank por país/cliente |
| Optimización | broadcast join, partitionBy, explain() |
| Delta Lake | Almacenamiento ACID en todas las capas |
| Structured Streaming | Ingesta incremental de nuevas transacciones |
| Medallion Architecture | Bronze → Silver → Gold |
| Validación | Separación válidos/inválidos con motivo de rechazo |
| Acumuladores | Métricas distribuidas de calidad |

---

## Preguntas de reflexión

1. ¿Qué formato elegiste para el almacenamiento final y por qué?
2. ¿Qué porcentaje de filas resultaron inválidas? ¿Qué tipo de error fue más frecuente?
3. ¿Por qué se usó `broadcast()` en el JOIN con productos?
4. ¿Qué información muestra `explain()` que ayuda a optimizar el pipeline?
5. ¿Cómo implementarías este pipeline en producción con Airflow?
6. Si los datos crecieran 100x, ¿qué cambiarías en el diseño?